The goal of this notebook is to create a comprehensive and robust workflow for registering CR and CL ptychography images. 

This notebook will load in two sets of tiff image stacks that correspond to positive and negative polarized images and then align them. Once aligned, the images may be summed to produce charge contrast or they can be subtracted to produce XMCD.

Then, the true zero will be determined by shifting the angle with the highest contrast in XMCD as 0. 

Finally, scans that fail a check for magnetic signal in the charge contrast will be removed from the stack with a suggestion to re-register the image. 

In [ ]:
## IMPORTS
import numpy as np
import os
import tifffile
from sklearn.cluster import KMeans
import matplotlib as mpl
import matplotlib.pyplot as plt
import cv2
#from tqdm import tqdm
from scipy.fft import fftn, ifftn
from scipy.ndimage import fourier_shift, gaussian_filter
from scipy.ndimage import shift as img_shift
from numpy.fft import fftshift
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
from scipy.optimize import curve_fit


from skimage.feature import graycomatrix, graycoprops
from skimage import io

import Functions as register


# Loading Projection Data

In [ ]:
# sample = "665"
# # Gamma = "Gammaneg45"
# # date = '_5AUG25_check_saturation_n50'
# # date1 = date
# Gamma = "Gamma0"
# date = '_3AUG25_check_saturation_n50'
# date1 = date



sample = "452"
Gamma = "Gamma90"

# sample = "105"
# Gamma = "Gamma0"
# date = '_17NOV25_auto_select_scans_fullrange'
# date1 = date

# sample = "105"
# Gamma = "Gamma45"
# date = '_19NOV25_select_fullrange'
# date1 = date

if sample == '452': 
    if Gamma == "Gamma45":
        date = '_16MAR26_og_ptycho'
        date1 = date
    
    
        data_dir = f"RawData/{sample}/{Gamma}/"
        
        pos_images = f"amplitude_pospol_stack{date}.tiff"
        pos_angles = f"angles_pospol{date1}.txt" #### angles are adjusted in ptychoMatFile_toTIFF
        neg_images = f"amplitude_negpol_stack{date}.tiff"
        neg_angles = f"angles_negpol{date1}.txt"  #### angles are adjusted in ptychoMatFile_toTIFF
        
        image_stack_pospol = tifffile.imread(os.path.join(data_dir, pos_images))
        image_stack_negpol = tifffile.imread(os.path.join(data_dir, neg_images))
        
        angles_pospol = np.loadtxt(os.path.join(data_dir, pos_angles))
        angles_negpol = np.loadtxt(os.path.join(data_dir, neg_angles))

        print("\n" + "=" * 60)
        i = 37
        print(f'delete index {i} from negpol')
        angles_negpol = np.delete(angles_negpol, i)
        image_stack_negpol = np.delete(image_stack_negpol, i, axis = 0)

        print('check shape of angle array and image array')
        print(angles_pospol.shape)
        print(angles_negpol.shape)
        print(image_stack_pospol.shape)
        print(image_stack_negpol.shape)
        print("\n" + "=" * 60)
        
    elif Gamma == 'Gamma90': 
        date = '_16MAR26_og_ptycho'
        date1 = date
        color_1 = 'm'
        color_2 = 'darkmagenta'
    
    
        data_dir = f"RawData/{sample}/{Gamma}/"
        
        pos_images = f"amplitude_pospol_stack{date}.tiff"
        pos_angles = f"angles_pospol{date1}.txt" #### angles are adjusted in ptychoMatFile_toTIFF
        neg_images = f"amplitude_negpol_stack{date}.tiff"
        neg_angles = f"angles_negpol{date1}.txt"  #### angles are adjusted in ptychoMatFile_toTIFF
        
        image_stack_pospol = tifffile.imread(os.path.join(data_dir, pos_images))
        image_stack_negpol = tifffile.imread(os.path.join(data_dir, neg_images))
        
        angles_pospol = np.loadtxt(os.path.join(data_dir, pos_angles))
        angles_negpol = np.loadtxt(os.path.join(data_dir, neg_angles))

        print("\n" + "=" * 60)
        i_pos = 29
        print(f'delete index {i_pos} from pospol, equals {angles_pospol[i_pos]} ')
        angles_pospol = np.delete(angles_pospol, i_pos)
        image_stack_pospol = np.delete(image_stack_pospol, i_pos, axis = 0)


        i_neg = 45
        print(f'delete index {i_neg} from negpol, equals {angles_negpol[i_neg]} ')
        angles_negpol = np.delete(angles_negpol, i_neg)
        image_stack_negpol = np.delete(image_stack_negpol, i_neg, axis = 0)
        
        i_both = [42, 43, 44]
 
        print(f'delete index {i_both} from both pospol and negpol')
        angles_negpol = np.delete(angles_negpol, i_both)
        image_stack_negpol = np.delete(image_stack_negpol, i_both, axis = 0)
        angles_pospol = np.delete(angles_pospol, i_both)
        image_stack_pospol = np.delete(image_stack_pospol, i_both, axis = 0)

        print('check shape of angle array and image array')
        print(angles_pospol.shape)
        print(angles_negpol.shape)
        print(image_stack_pospol.shape)
        print(image_stack_negpol.shape)
        print("\n" + "=" * 60)


    elif Gamma == 'Gamma0': 
        date = '_16MAR26_og_ptycho'
        date1 = date
    
    
        data_dir = f"RawData/{sample}/{Gamma}/"
        
        pos_images = f"amplitude_pospol_stack{date}.tiff"
        pos_angles = f"angles_pospol{date1}.txt" #### angles are adjusted in ptychoMatFile_toTIFF
        neg_images = f"amplitude_negpol_stack{date}.tiff"
        neg_angles = f"angles_negpol{date1}.txt"  #### angles are adjusted in ptychoMatFile_toTIFF
        
        image_stack_pospol = tifffile.imread(os.path.join(data_dir, pos_images))
        image_stack_negpol = tifffile.imread(os.path.join(data_dir, neg_images))
        
        angles_pospol = np.loadtxt(os.path.join(data_dir, pos_angles))
        angles_negpol = np.loadtxt(os.path.join(data_dir, neg_angles))

else: 
    data_dir = f"RawData/{sample}/{Gamma}/"
    
    pos_images = f"OD_pospol_stack{date}.tiff"
    print(pos_images)
    #pos_phase_images = f"phase_pospol_stack{date}.tiff"
    pos_angles = f"angles_pospol{date1}.txt" #### angles are adjusted in ptychoMatFile_toTIFF
    neg_images = f"OD_negpol_stack{date}.tiff"
    #neg_phase_images = f"phase_negpol_stack{date}.tiff"
    neg_angles = f"angles_negpol{date1}.txt"  #### angles are adjusted in ptychoMatFile_toTIFF
    
    image_stack_pospol = tifffile.imread(os.path.join(data_dir, pos_images))
    image_stack_negpol = tifffile.imread(os.path.join(data_dir, neg_images))
    # phase_stack_pospol = tifffile.imread(os.path.join(data_dir, pos_phase_images))
    # phase_stack_negpol = tifffile.imread(os.path.join(data_dir, neg_phase_images))
    
    angles_pospol = np.loadtxt(os.path.join(data_dir, pos_angles))
    angles_negpol = np.loadtxt(os.path.join(data_dir, neg_angles))

In [ ]:
print(angles_pospol.shape)
print(angles_negpol.shape)

In [ ]:
for k in np.arange(0, image_stack_pospol.shape[0]):
    print(f'======================= {k} =======================')
    print(f' positive angle = {angles_pospol[k]}')
    print(f' negative angle = {angles_negpol[k]}')
    if angles_negpol[k] - angles_pospol[k] > 0.5: 
        print(f'CHECK INDEX {k}')

    elif angles_negpol[k+1] - angles_negpol[k] < 0.5: 
        print(f'REPEAT SCAN {k}')
    else: 
        continue


In [ ]:
# for k in range(image_stack_pospol.shape[0]):
#     print('##################') 
#     print(f'k = {k}')
#     print('##################') 
#     ## Plot left, right, and charge contrast 
#     norm = mpl.colors.Normalize(vmin=0, vmax=1)
#     fig_size = 3.5
#     fig, axes = plt.subplots(1, 2, figsize=(fig_size * 3, fig_size))
#     left = axes[0].imshow(image_stack_pospol[k], norm=norm)
#     axes[0].set_title(f" Right polarization \n  α = {angles_pospol[k]:.2f}", fontsize=14) 
#     fig.colorbar(left)
#     right = axes[1].imshow(image_stack_negpol[k], norm=norm)
#     axes[1].set_title(f" Left polarization \n α = {angles_negpol[k]:.2f}",  fontsize=14)
#     fig.colorbar(right)
#     plt.show()


# Xcorr plot single scans

1. Creates a mask based on the average intensity of the holes and normalizes incoming intensity to 1
2. Performs cross correlation of the fiducials over a range of 10 pixels
3. translates the negative polarity image
4. XMCD = translated left - right
5. CC = translated left + right 


# Single scan

In [ ]:
######## Look at a single scan, select k ########
k = 0 #51
#################################################

angle = angles_pospol[k]


## Plot left, right, and charge contrast 
norm = mpl.colors.Normalize(vmin=0, vmax=2)
fig_size = 3.5
fig, axes = plt.subplots(1, 2, figsize=(fig_size * 3, fig_size))
left = axes[0].imshow(image_stack_negpol, norm=norm)
axes[0].set_title(f" Left polarization \n Γ = 0, α = {angle:.2f}", fontsize=14) 
fig.colorbar(left)
right = axes[1].imshow(image_stack_pospol, norm=norm)
axes[1].set_title(f" Right polarization \n Γ = 0, α = {angle:.2f}",  fontsize=14)
fig.colorbar(right)
plt.show()

## Plot masked left and right scans
norm = mpl.colors.Normalize(vmin=0, vmax=4)
fig_size = 3.5
fig, axes = plt.subplots(1, 1, figsize=(fig_size * 3, fig_size))
pos = axes.imshow(image_stack_negpol + image_stack_pospol, norm=norm)
axes.set_title(f"CPL + CPR = CC \n unaligned") 
fig.colorbar(pos)
plt.show()

norm = mpl.colors.Normalize(vmin=-0.5, vmax=0.5)
fig_size = 3.5
fig, axes = plt.subplots(1, 1, figsize=(fig_size * 3, fig_size))
pos = axes.imshow(image_stack_negpol - image_stack_pospol, norm=norm)
axes.set_title(f"CPL - CPR = XMCD \n unaligned") 
fig.colorbar(pos)
plt.show()

## Plot original histogram of left and right
title = 'Un-normalized'
color_neg = 'blue'
color_pos = 'orange'
vals_neg = image_stack_negpol.flatten()
vals_pos = image_stack_pospol.flatten()
plot_histogram(vals_neg, vals_pos, color_neg, color_pos)



## normalize left and right to average intensity of the fiducials 
norm_neg = image_stack_negpol/(np.max(image_stack_negpol))
norm_pos = image_stack_pospol/(np.max(image_stack_pospol))


## Plot left, right, and charge contrast 
norm = mpl.colors.Normalize(vmin=-2, vmax=2)
fig_size = 3.5
fig, axes = plt.subplots(1, 2, figsize=(fig_size * 3, fig_size))
left = axes[0].imshow(-1*np.log(norm_neg), norm=norm)
#axes[0].set_title(f" {Gamma} Left polarization @ {angle:.2f} \n unnormalized") 
axes[0].set_title(f"Optical Density Left polarization \n Γ = 0, α = {angle:.2f}", fontsize=14) 
fig.colorbar(left)
right = axes[1].imshow(-1*np.log(norm_pos), norm=norm)
#axes[1].set_title(f"{Gamma} Right polarization @ {angle:.2f} \n unnormalized")
axes[1].set_title(f" Optical Density Right polarization \n Γ = 0, α = {angle:.2f}",  fontsize=14)
fig.colorbar(right)
plt.show()


## Plot masked left and right scans
norm = mpl.colors.Normalize(vmin=0, vmax=4)
fig_size = 3.5
fig, axes = plt.subplots(1, 1, figsize=(fig_size * 3, fig_size))
pos = axes.imshow(-1*np.log(norm_pos) + -1*np.log(norm_neg), norm=norm)
axes.set_title(f"Charge Contrast \n unaligned") 
fig.colorbar(pos)
plt.show()


# ## Plot normalized histogram left and right
# title = 'Normalized'
# vals_neg = norm_neg.flatten()
# vals_pos = norm_pos.flatten()
# plot_histogram(vals_neg, vals_pos, color_neg, color_pos)

# ## apply a mask to 30% of hole intenstity 
# factor = 0.2 #30%
# masked_pos, masked_neg = mask_holes_to_one(norm_pos,norm_neg, factor)

# ## Plot masked left and right scans
# norm = mpl.colors.Normalize(vmin=-0.01, vmax=0.01)
# fig_size = 3.5
# fig, axes = plt.subplots(1, 2, figsize=(fig_size * 3, fig_size))
# pos = axes[0].imshow(masked_pos, norm=norm)
# axes[0].set_title(f"{Gamma} positive @ {angle:.2f} \n masked") 
# fig.colorbar(pos)
# neg = axes[1].imshow(masked_neg, norm=norm)
# axes[1].set_title(f"{Gamma} negative @ {angle:.2f} \n masked")
# fig.colorbar(neg)
# plt.show()

## Perform crosscorrelation on normalized/masked values

masked_pos = -1*np.log(norm_pos)
masked_neg = -1*np.log(norm_neg)
print(f'shape of images = {np.shape(masked_pos)} and {np.shape(masked_neg)}')

bound = 50
translation, corr = align_images_CC(masked_pos, masked_neg, bound)
print(f"Estimated Translation = {translation}")

## Translating the Negative Image
f_target = fftn(norm_neg)
f_target_shifted = fourier_shift(f_target, translation)
translated_negpol = ifftn(f_target_shifted).real


unaligned_diffMap = -1*np.log(norm_neg) - -1*np.log(norm_pos)
aligned_diffMap = -1*np.log(translated_negpol) - -1*np.log(norm_pos)

plot_XMCD_crosscorr(unaligned_diffMap, aligned_diffMap, corr)

## Plot masked left and right scans
norm = mpl.colors.Normalize(vmin=0, vmax=4)
fig_size = 3.5
fig, axes = plt.subplots(1, 1, figsize=(fig_size * 3, fig_size))
pos = axes.imshow(-1*np.log(translated_negpol) + -1*np.log(norm_pos), norm=norm)
axes.set_title(f"Charge Contrast aligned \n Γ=0 @ {angle:.2f}") 
fig.colorbar(pos)
plt.show()

In [ ]:
## Plot masked left and right scans
norm = mpl.colors.Normalize(vmin=0, vmax=4)
fig_size = 3.5
fig, axes = plt.subplots(1, 1, figsize=(fig_size * 3, fig_size))
pos = axes.imshow(-1*np.log(translated_negpol) + -1*np.log(norm_pos), norm=norm)
axes.set_title(f"Co Charge Contrast aligned \n Γ=0 @ {angle:.2f}") 
fig.colorbar(pos)
plt.show()

In [ ]:
## Plot masked left and right scans
norm = mpl.colors.Normalize(vmin=-0.75, vmax=0.75)
fig_size = 3.5
fig, axes = plt.subplots(1, 1, figsize=(fig_size * 3, fig_size))
pos = axes.imshow(-1*np.log(translated_negpol) - -1*np.log(norm_pos), norm=norm)
axes.set_title(f"Co XMCD aligned \n Γ=0 @ {angle:.2f}") 
fig.colorbar(pos)
plt.show()

In [ ]:
stop

# Stack

In [ ]:
######## Look at a single scan, select k ########
k = 8 #51
#################################################

angle = angles_pospol[k]
print(angles_pospol[k])
print(angles_negpol[k])

## Plot left, right, and charge contrast 
norm = mpl.colors.Normalize(vmin=0, vmax=2)
fig_size = 3.5
fig, axes = plt.subplots(1, 2, figsize=(fig_size * 3, fig_size))
left = axes[0].imshow(image_stack_negpol[k], norm=norm)
axes[0].set_title(f" Left polarization \n Γ = 0, α = {angle:.2f}", fontsize=14) 
fig.colorbar(left)
right = axes[1].imshow(image_stack_pospol[k], norm=norm)
axes[1].set_title(f" Right polarization \n Γ = 0, α = {angle:.2f}",  fontsize=14)
fig.colorbar(right)
plt.show()

## Plot original histogram of left and right
title = 'Un-normalized'
color_neg = 'blue'
color_pos = 'orange'
vals_neg = image_stack_negpol[k].flatten()
vals_pos = image_stack_pospol[k].flatten()
plot_histogram(vals_neg, vals_pos, color_neg, color_pos)

## Plot original histogram of left and right
title = 'Un-normalized, cropped around fiducial'
crop_neg = image_stack_negpol[k][325:375, 190:240]
crop_pos = image_stack_pospol[k][325:375, 190:240]

print(' max image')
print(np.max(crop_pos))

norm = mpl.colors.Normalize(vmin=0, vmax=2)
fig_size = 3.5
fig, axes = plt.subplots(1, 2, figsize=(fig_size * 3, fig_size))
left = axes[0].imshow(crop_neg, norm=norm)
axes[0].set_title(f" Left polarization \n Γ = 0, α = {angle:.2f}", fontsize=14) 
fig.colorbar(left)
right = axes[1].imshow(crop_pos, norm=norm)
axes[1].set_title(f" Right polarization \n Γ = 0, α = {angle:.2f}",  fontsize=14)
fig.colorbar(right)
plt.show()


color_neg = 'blue'
color_pos = 'orange'
vals_neg = crop_neg.flatten()
vals_pos = crop_pos.flatten()
print(' max flattened')
print(np.max(vals_pos))
plot_histogram(vals_neg, vals_pos, color_neg, color_pos)

## Plot masked left and right scans
norm = mpl.colors.Normalize(vmin=0, vmax=4)
fig_size = 3.5
fig, axes = plt.subplots(1, 1, figsize=(fig_size * 3, fig_size))
pos = axes.imshow(image_stack_negpol[k] + image_stack_pospol[k], norm=norm)
axes.set_title(f"CPL + CPR = CC \n unaligned") 
fig.colorbar(pos)
plt.show()

norm = mpl.colors.Normalize(vmin=-0.5, vmax=0.5)
fig_size = 3.5
fig, axes = plt.subplots(1, 1, figsize=(fig_size * 3, fig_size))
pos = axes.imshow(image_stack_negpol[k] - image_stack_pospol[k], norm=norm)
axes.set_title(f"CPL - CPR = XMCD \n unaligned") 
fig.colorbar(pos)
plt.show()


## normalize left and right to average intensity of the fiducials 
norm_neg, fid_avg_neg = auto_normalize_by_fiducial(image_stack_negpol[k], color_neg)
norm_pos, fid_avg_pos = auto_normalize_by_fiducial(image_stack_pospol[k], color_pos)

# print(f'norm_neg = {norm_neg}')
# print(f'norm_pos = {norm_pos}')

## Plot left, right, and charge contrast 
norm = mpl.colors.Normalize(vmin=-2, vmax=2)
fig_size = 3.5
fig, axes = plt.subplots(1, 2, figsize=(fig_size * 3, fig_size))
left = axes[0].imshow(-1*np.log(norm_neg), norm=norm)
#axes[0].set_title(f" {Gamma} Left polarization @ {angle:.2f} \n unnormalized") 
axes[0].set_title(f"Optical Density Left polarization \n Γ = 0, α = {angle:.2f}", fontsize=14) 
fig.colorbar(left)
right = axes[1].imshow(-1*np.log(norm_pos), norm=norm)
#axes[1].set_title(f"{Gamma} Right polarization @ {angle:.2f} \n unnormalized")
axes[1].set_title(f" Optical Density Right polarization \n Γ = 0, α = {angle:.2f}",  fontsize=14)
fig.colorbar(right)
plt.show()


## Plot masked left and right scans
norm = mpl.colors.Normalize(vmin=0, vmax=4)
fig_size = 3.5
fig, axes = plt.subplots(1, 1, figsize=(fig_size * 3, fig_size))
pos = axes.imshow(-1*np.log(norm_pos) + -1*np.log(norm_neg), norm=norm)
axes.set_title(f"Charge Contrast \n unaligned") 
fig.colorbar(pos)
plt.show()


## Plot normalized histogram left and right
title = 'Normalized'
vals_neg = norm_neg.flatten()
vals_pos = norm_pos.flatten()
plot_histogram(vals_neg, vals_pos, color_neg, color_pos)

## apply a mask to 30% of hole intenstity 
factor = 0.7 #30%
masked_pos, masked_neg = mask_holes_to_one(norm_pos,norm_neg, factor)

## Plot masked left and right scans
norm = mpl.colors.Normalize(vmin=-0.01, vmax=0.01)
fig_size = 3.5
fig, axes = plt.subplots(1, 2, figsize=(fig_size * 3, fig_size))
pos = axes[0].imshow(masked_pos, norm=norm)
axes[0].set_title(f"{Gamma} positive @ {angle:.2f} \n masked") 
fig.colorbar(pos)
neg = axes[1].imshow(masked_neg, norm=norm)
axes[1].set_title(f"{Gamma} negative @ {angle:.2f} \n masked")
fig.colorbar(neg)
plt.show()


## Perform crosscorrelation on normalized/masked values
bound = 150
translation, corr = align_images_CC(masked_pos, masked_neg, bound)
print(f"Estimated Translation = {translation}")

## Translating the Negative Image
f_target = fftn(norm_neg)
f_target_shifted = fourier_shift(f_target, translation)
translated_negpol = ifftn(f_target_shifted).real

## Translating the phase image by the same amount
# f_target = fftn(phase_stack_negpol[k])
# f_target_shifted = fourier_shift(f_target, translation)
# translated_negpol_phase = ifftn(f_target_shifted).real

unaligned_diffMap = -1*np.log(norm_neg) - -1*np.log(norm_pos)
aligned_diffMap = -1*np.log(translated_negpol) - -1*np.log(norm_pos)

plot_XMCD_crosscorr(unaligned_diffMap, aligned_diffMap, corr)

## Plot masked left and right scans
norm = mpl.colors.Normalize(vmin=0, vmax=4)
fig_size = 3.5
fig, axes = plt.subplots(1, 1, figsize=(fig_size * 3, fig_size))
pos = axes.imshow(-1*np.log(translated_negpol) + -1*np.log(norm_pos), norm=norm)
axes.set_title(f"Charge Contrast aligned \n Γ=0 @ {angle:.2f}") 
fig.colorbar(pos)
plt.show()


    

In [ ]:
for k in np.arange(1, image_stack_pospol.shape[0]):
    print(angles_pospol[k])
    print(angles_negpol[k])
    angle = angles_pospol[k]
    plot_CLCR(image_stack_pospol[k], image_stack_negpol[k])

In [ ]:
######## Look at a single scan, select k ########
k = 38
#################################################

angle = angles_pospol[k]

# transition everything to OD
OD_img_pos = -1*np.log(image_stack_negpol[k])
OD_img_neg = -1*np.log(image_stack_pospol[k])

## Plot left, right OD un-normalized
norm = mpl.colors.Normalize(vmin=-1, vmax=1)
fig_size = 3.5
fig, axes = plt.subplots(1, 2, figsize=(fig_size * 3, fig_size))
left = axes[0].imshow(OD_img_neg, norm=norm)
#axes[0].set_title(f" {Gamma} Left polarization @ {angle:.2f} \n unnormalized") 
axes[0].set_title(f" Left polarization \n {Gamma}, α = {angle:.2f}", fontsize=14) 
fig.colorbar(left)
right = axes[1].imshow(OD_img_pos, norm=norm)
#axes[1].set_title(f"{Gamma} Right polarization @ {angle:.2f} \n unnormalized")
axes[1].set_title(f" Right polarization \n {Gamma}, α = {angle:.2f}",  fontsize=14)
fig.colorbar(right)
plt.show()

## CC
norm = mpl.colors.Normalize(vmin=-1, vmax=1)
fig_size = 3.5
fig, axes = plt.subplots(1, 1, figsize=(fig_size * 3, fig_size))
pos = axes.imshow(OD_img_pos + OD_img_neg, norm=norm)
axes.set_title(f"{Gamma} positive @ {angle:.2f} \n CC") 
fig.colorbar(pos)
plt.show()


## Plot original histogram of left and right
title = 'Un-normalized'
color_neg = 'blue'
color_pos = 'orange'
vals_neg = OD_img_neg.flatten()
vals_pos = OD_img_pos.flatten()
plot_histogram(vals_neg, vals_pos, color_neg, color_pos)

###################################### Normalized

## normalize left and right to average intensity of the fiducials 
norm_neg, fid_avg_neg = auto_normalize_by_fiducial(OD_img_neg, color_neg)
norm_pos, fid_avg_pos = auto_normalize_by_fiducial(OD_img_pos, color_pos)

## Plot left, right, and charge contrast 
norm = mpl.colors.Normalize(vmin=-2, vmax=2)
fig_size = 3.5
fig, axes = plt.subplots(1, 2, figsize=(fig_size * 3, fig_size))
left = axes[0].imshow((norm_neg), norm=norm)
#axes[0].set_title(f" {Gamma} Left polarization @ {angle:.2f} \n unnormalized") 
axes[0].set_title(f"Normalized Left polarization \n {Gamma}, α = {angle:.2f}", fontsize=14) 
fig.colorbar(left)
right = axes[1].imshow((norm_pos), norm=norm)
#axes[1].set_title(f"{Gamma} Right polarization @ {angle:.2f} \n unnormalized")
axes[1].set_title(f" Normalized Right polarization \n {Gamma}, α = {angle:.2f}",  fontsize=14)
fig.colorbar(right)
plt.show()

## Plot normalized histogram left and right
title = 'Normalized'
vals_neg = norm_neg.flatten()
vals_pos = norm_pos.flatten()
plot_histogram(vals_neg, vals_pos, color_neg, color_pos)

norm_CC_neg45 = (norm_neg) + (norm_pos)
norm_XMCD_neg45 = (norm_neg - norm_pos)

## Plot left, right, and charge contrast 
norm = mpl.colors.Normalize(vmin=-2, vmax=2)
fig_size = 3.5
fig, axes = plt.subplots(1, 2, figsize=(fig_size * 3, fig_size))
left = axes[0].imshow(norm_CC_neg45, norm=norm)
axes[0].set_title(f"Normalized CC \n {Gamma}, α = {angle:.2f}", fontsize=14) 
fig.colorbar(left)
right = axes[1].imshow(norm_XMCD_neg45, norm=norm)
axes[1].set_title(f" Normalized XMCD \n {Gamma}, α = {angle:.2f}",  fontsize=14)
fig.colorbar(right)
plt.show()


################################ Histograms of CC 

## Plot normalized histogram left and right
title = 'Normalized CC'
vals_CC_neg45 = norm_CC_neg45.flatten()

# plot_single_histogram(vals_CC_0, color_pos)


# XCorr run all scans

In [ ]:
## Run full alpha scan, no background subtraction

## Initialize arrays 
translated_negpol = np.zeros_like(image_stack_negpol)
# translated_negpol_phase = np.zeros_like(phase_stack_negpol)

norm_pos_OD = np.zeros_like(image_stack_pospol)
norm_neg_OD = np.zeros_like(image_stack_negpol)

unaligned_error_norm = np.empty(image_stack_pospol.shape[0])
aligned_error_norm = np.empty(image_stack_pospol.shape[0])

## Initialize other parameters
# colors needed for function :/
color_neg = 'blue'
color_pos = 'orange'
factor = 0.7 #30%
bound = 200 ## Add if statements later on for scans that need larger bounds

for k in range(image_stack_pospol.shape[0]):
    
    angle = angles_pospol[k]

    norm_neg, fid_avg_neg = auto_normalize_by_fiducial(image_stack_negpol[k],color_neg, plot_hist = False)
    norm_pos, fid_avg_pos = auto_normalize_by_fiducial(image_stack_pospol[k],color_pos, plot_hist = False)

    ## apply a mask to 30% of hole intenstity 
    masked_pos, masked_neg = mask_holes_to_one(norm_pos, norm_neg, factor)

    ## Perform crosscorrelation on normalized/masked values
    translation, corr = align_images_CC(masked_pos, masked_neg, bound)
    
    # Translating the Image
    f_target = fftn(norm_neg)
    f_target_shifted = fourier_shift(f_target, translation)
    translated_negpol_OD = ifftn(f_target_shifted).real
    

    # # Translating the phase image by the same amount
    # f_target = fftn(phase_stack_negpol[k])
    # f_target_shifted = fourier_shift(f_target, translation)
    # translated_negpol_phase[k] = ifftn(f_target_shifted).real
    
    ## XMCD
    # unaligned_diffMap = norm_neg_OD[k] - norm_pos_OD[k]
    # aligned_diffMap = translated_negpol[k] - norm_pos_OD[k]

    norm_pos_OD[k] = -1*np.log(norm_pos)
    norm_neg_OD[k] = -1*np.log(norm_neg)
    translated_negpol[k] =-1*np.log(translated_negpol_OD)

    unaligned_diffMap = -1*np.log(norm_neg) - (-1*np.log(norm_pos))
    aligned_diffMap = -1*np.log(translated_negpol_OD) - (-1*np.log(norm_pos))
    # unaligned_diffMap = -1*np.log(norm_neg) - -1*np.log(norm_pos)
    # aligned_diffMap = -1*np.log(translated_negpol) - -1*np.log(norm_pos)

    plot_XMCD_crosscorr(unaligned_diffMap, aligned_diffMap, corr)

    # Evaluating the transformation's alignment by comparing the difference image before and after alignment
    orig_error_mask = calc_NRMSE(image_stack_negpol[k], image_stack_pospol[k])
    align_error_mask = calc_NRMSE(translated_negpol[k], norm_pos_OD[k])

    unaligned_error_norm[k] = orig_error_mask
    aligned_error_norm[k] = align_error_mask
    
    print(f"Error Before = {orig_error_mask*100:.2f}%\nError After = {align_error_mask*100:.2f}%")



In [ ]:
# Plotting I0 over angle

fid_amp_neg = [ ]
fid_amp_pos = [ ]

for k in range(image_stack_pospol.shape[0]): 
    angle =  angles_pospol[k]

    ___, fid_avg_neg = auto_normalize_by_fiducial(image_stack_negpol[k], color_neg, plot_hist = False)
    ___, fid_avg_pos = auto_normalize_by_fiducial(image_stack_pospol[k], color_pos, plot_hist = False)

    fid_amp_neg.append(fid_avg_neg)
    fid_amp_pos.append(fid_avg_pos)

plt.plot(angles_negpol, fid_amp_neg, marker = 'o', label = 'CPL')
plt.plot(angles_pospol, fid_amp_pos, marker = 'o', label = 'CPL')
plt.xlabel('angle deg.')
plt.ylabel('I0')
plt.legend()
plt.title(f'I0 vs angle {Gamma}')

plt.show()

    
    

In [ ]:
OD_chrgcntrst= norm_pos_OD + translated_negpol
OD_XMCD= translated_negpol- norm_pos_OD

XMCD_unaligned = image_stack_negpol - image_stack_pospol
CC_unaligned = image_stack_negpol + image_stack_pospol

In [ ]:
k = 8
plt.imshow(OD_chrgcntrst[k], vmin = 3, vmax = 4)
plt.title(angles_pospol[k])

In [ ]:
for k in np.arange(1, image_stack_pospol.shape[0]):
    angle = angles_pospol[k]
    plot_XMCD_CC_normalizeCC(OD_XMCD[k], OD_chrgcntrst[k], angle)


In [ ]:
stop

In [ ]:
k = 10

In [ ]:
## Select different 50x50 regions to plot and calculate the mean and variance of these regions for 1 scan

# averaging areas
half_size = 60
shifted_angles = angles_pospol
height, width = image_stack_pospol.shape[1:]
center_y = np.array([60, 300, 475, 60,  height // 2-50, 475, 60,  300, 475])
center_x = np.array([60, 60, 60,   300, width // 2-50,  200, 475, 475, 475])
i = 4

# normalize arrays to store mean and variance over the averaging areas
mean_region = np.zeros(np.shape(center_y)[0])
variance_region = np.zeros(np.shape(center_y)[0])
mean_ua_region = np.zeros(np.shape(center_y)[0])
variance_ua_region = np.zeros(np.shape(center_y)[0])

color_cc = 'green'
range_min, range_max, norm_CC, fid_avg_CC = auto_normalize_charge(OD_chrgcntrst[k], color_cc)

for k in range(image_stack_pospol.shape[0]):
    print(shifted_angles[k])
    
    # Slice
    if k ==1: 
        shift_y = 0
        shift_x = 220
        cropped_XMCD = OD_XMCD[k][center_y[i] + shift_y - half_size :center_y[i]+ shift_y + half_size,
                      center_x[i] + shift_x - half_size:center_x[i] + shift_x + half_size]
        cropped_CC = OD_chrgcntrst[k][center_y[i] + shift_y - half_size:center_y[i] + shift_y + half_size,
                              center_x[i] + shift_x - half_size:center_x[i] + shift_x + half_size]
    
        # ua stands for unaligned
        cropped_XMCD_ua = XMCD_unaligned[k][center_y[i] - half_size:center_y[i] + half_size,
                              center_x[i] - half_size:center_x[i] + half_size]
        cropped_CC_ua = CC_unaligned[k][center_y[i] - half_size:center_y[i] + half_size,
                              center_x[i] - half_size:center_x[i] + half_size]
    elif k == 2: 
        shift_y = 60
        shift_x = 150
        cropped_XMCD = OD_XMCD[k][center_y[i] + shift_y - half_size :center_y[i]+ shift_y + half_size,
                      center_x[i] + shift_x - half_size:center_x[i] + shift_x + half_size]
        cropped_CC = OD_chrgcntrst[k][center_y[i] + shift_y - half_size:center_y[i] + shift_y + half_size,
                              center_x[i] + shift_x - half_size:center_x[i] + shift_x + half_size]
    
        # ua stands for unaligned
        cropped_XMCD_ua = XMCD_unaligned[k][center_y[i] - half_size:center_y[i] + half_size,
                              center_x[i] - half_size:center_x[i] + half_size]
        cropped_CC_ua = CC_unaligned[k][center_y[i] - half_size:center_y[i] + half_size,
                              center_x[i] - half_size:center_x[i] + half_size]
    
    elif k ==0: 
        shift_y = 0
        shift_x = 120

        cropped_XMCD = OD_XMCD[k][center_y[i] + shift_y - half_size :center_y[i]+ shift_y + half_size,
                      center_x[i] + shift_x - half_size:center_x[i] + shift_x + half_size]
        cropped_CC = OD_chrgcntrst[k][center_y[i] + shift_y - half_size:center_y[i] + shift_y + half_size,
                              center_x[i] + shift_x - half_size:center_x[i] + shift_x + half_size]
    
        # ua stands for unaligned
        cropped_XMCD_ua = XMCD_unaligned[k][center_y[i] - half_size:center_y[i] + half_size,
                              center_x[i] - half_size:center_x[i] + half_size]
        cropped_CC_ua = CC_unaligned[k][center_y[i] - half_size:center_y[i] + half_size,
                              center_x[i] - half_size:center_x[i] + half_size]
        
    else:
        cropped_XMCD = OD_XMCD[k][center_y[i] - half_size:center_y[i] + half_size,
                              center_x[i] - half_size:center_x[i] + half_size]
        cropped_CC = OD_chrgcntrst[k][center_y[i] - half_size:center_y[i] + half_size,
                              center_x[i] - half_size:center_x[i] + half_size]
    
        # ua stands for unaligned
        cropped_XMCD_ua = XMCD_unaligned[k][center_y[i] - half_size:center_y[i] + half_size,
                              center_x[i] - half_size:center_x[i] + half_size]
        cropped_CC_ua = CC_unaligned[k][center_y[i] - half_size:center_y[i] + half_size,
                              center_x[i] - half_size:center_x[i] + half_size]
    
    mean_region[i] = np.mean(cropped_CC)
    variance_region[i] = np.var(cropped_CC)
    mean_ua_region[i] = np.mean(cropped_CC_ua)
    variance_ua_region[i] = np.var(cropped_CC_ua)

    # Plot the images to see what each section looks like 
    norm_XMCD = mpl.colors.Normalize(vmin=-0.2, vmax=0.2)
    norm_charge = mpl.colors.Normalize(vmin=1, vmax=3)
    fig_size = 4.5
    fig, axes = plt.subplots(1, 2, figsize=(fig_size * 2, fig_size))
    XMCD_OD = axes[0].imshow(cropped_XMCD,cmap = 'viridis', norm=norm_XMCD)
    #axes[0].set_title(f"XMCD @ {shifted_angles[k]:.2f} \n averaging area is {center_y[i]},{center_x[i]}") 
    fig.colorbar(XMCD_OD)
    Charge_OD = axes[1].imshow(cropped_CC,cmap = 'viridis', norm=norm_charge)
    #axes[1].set_title(f"Charge constrast @ {shifted_angles[k]:.2f} \n averaging area is {center_y[i]},{center_x[i]}")
    fig.colorbar(Charge_OD)
    plt.show()


In [ ]:
date

In [ ]:
modifier = '_test'

In [ ]:
## Optical Density
params = 'test'
tifffile.imwrite(os.path.join(data_dir, f"OD_negpol_aligned_stack{date}_{Gamma}{modifier}.tiff"), translated_negpol)### CHANGE BACK TO FINAL

tifffile.imwrite(os.path.join(data_dir, f"OD_chrgcntrst_stack{date}_{Gamma}{modifier}.tiff"), OD_chrgcntrst)### CHANGE BACK TO FINAL
tifffile.imwrite(os.path.join(data_dir, f"OD_XMCD_stack{date}_{Gamma}{modifier}.tiff"), OD_XMCD) ### CHANGE BACK TO FINAL


# Plot NRMSE (aka texture plots)

In [ ]:
angles_pospol

In [ ]:
index = range(image_stack_pospol.shape[0])
plt.plot(angles_pospol, aligned_error_norm, marker = 'o', color = 'm', label = 'aligned_error')
plt.ylabel('NRMSE (%)')
plt.xlabel('index')
plt.title(f'{Gamma}, normalized to holes')
plt.legend()
plt.show()

# calculate the angle shift

In [ ]:
def cos_fit(x, A, yo, phase): 
    # x = theta
    b_fix = np.pi/ np.ptp(x)
    return A*np.cos(b_fix*x+phase) + yo

In [ ]:
plot_angles = np.linspace(angles_pospol[0], np.max(angles_pospol), angles_pospol.shape[0])
plot_angles_rad = np.deg2rad(plot_angles)

bounds = ([0, 0, -2*np.pi], [np.inf, np.inf, 2*np.pi])
guess = [0.1, 2*np.mean(aligned_error_norm), 6]
print(2*np.mean(aligned_error_norm))

y_params, y_cov = curve_fit(cos_fit, plot_angles_rad, aligned_error_norm, p0=guess, bounds=bounds)
print(y_params)
y_cos_fit = cos_fit(plot_angles_rad, y_params[0], y_params[1], y_params[2])
b_fix = np.pi/ np.ptp(plot_angles_rad) ## period  = 2pi/B

plt.plot(angles_pospol, aligned_error_norm, marker = 'o', color = color_1, label = 'NRMSE data')
plt.plot(angles_pospol, y_cos_fit, color = color_2, linewidth = '3', label = f'cos fit\n y = {y_params[0]:.2f}*cos({b_fix:.2f}*angle + {y_params[2]:.2f}) + {y_params[1]:.2f}')
#plt.axvline(angles_pospol[52], color = 'k')
plt.plot(angles_pospol[(np.argmax(y_cos_fit))],(np.max(y_cos_fit)),'ro', label = f'Max at {angles_pospol[(np.argmax(y_cos_fit))]:.2f}') 

plt.xlabel('tilt angle (deg)')
plt.ylabel('NRMSE (%)')
plt.title(f'fit 0 deg. by shifting {angles_pospol[(np.argmax(y_cos_fit))]:.2f} deg \n {Gamma} unshifted')
plt.legend(loc = 'lower right')
plt.show()
#m and darkmagenta
#cornflowerblue and royalblue

In [ ]:
shifted_angles = angles_pospol - angles_pospol[(np.argmax(y_cos_fit))]
print(shifted_angles[0])
print(angles_pospol[0])

plot_angles_shift = np.linspace(shifted_angles[0], np.max(shifted_angles), shifted_angles.shape[0])
plot_angles_rad_shift = np.deg2rad(plot_angles_shift)


bounds = ([0, 0, -2*np.pi], [np.inf, np.inf, 2*np.pi])
guess = [5, np.mean(aligned_error_norm), 2]

y_params_shift, y_cov = curve_fit(cos_fit, plot_angles_rad_shift, aligned_error_norm, p0=guess, bounds=bounds)
y_cos_fit_shift = cos_fit(plot_angles_rad_shift, y_params_shift[0], y_params_shift[1], y_params_shift[2])
b_fix = np.pi/ np.ptp(plot_angles_rad_shift) ## period  = 2pi/B


plt.plot(shifted_angles, aligned_error_norm, marker = 'o', color = color_1, label = 'shifted NRMSE data')
plt.plot(shifted_angles, y_cos_fit_shift, color = color_2, linewidth = '3', label = f'cos fit \n y = {y_cos_fit_shift[0]:.2f}*cos({b_fix:.2f}*angle + {y_cos_fit_shift[2]:.2f}) + {y_cos_fit_shift[1]:.2f}')
# plt.axvline(angles_pospol[52], color = 'k')
plt.plot(shifted_angles[(np.argmax(y_cos_fit_shift))],(np.max(y_cos_fit_shift)),'ro', label = f'Max at {shifted_angles[(np.argmax(y_cos_fit))]:.2f}') 

plt.xlabel('tilt angle (deg)')
plt.ylabel('NRMSE (%)')
plt.title(f'Fit 0 deg., shifted \n {Gamma}')
plt.legend(loc = 'lower left')
plt.show()

In [ ]:
# np.savetxt(os.path.join(data_dir, f"angles_shifted{date}.txt"), shifted_angles)

In [ ]:
stop

# quantifying error with CC FWHM

# quantifying error with variance

In [ ]:
## Plot CC

k_good = 29
k_question = 17
k_bad = 25  # selected from scans to check, avg, from below
title = f'Compare CC histogram for good and bad scan'
vals_CC_good = OD_chrgcntrst[k_good].flatten()
vals_CC_bad = OD_chrgcntrst[k_bad].flatten()
vals_CC_q= OD_chrgcntrst[k_question].flatten()

plt.hist(vals_CC_good, bins = 100, alpha = 0.65, color = 'tab:green', label = f'Good Scan @ α = {angles_pospol[k_good]:.2f}')
plt.hist(vals_CC_bad, bins = 100, alpha = 0.5, color = 'tab:red', label = f'Bad Scan @ α = {angles_pospol[k_bad]:.2f}')
plt.hist(vals_CC_q, bins = 100, alpha = 0.5, color = 'tab:blue', label = f'Questionable Scan @ α = {angles_pospol[k_question]:.2f}')

plt.xlabel('Intensity')
plt.ylabel('Pixel Count')
# plt.xlim(-0.25, 0.25)
# plt.ylim(0, 1500)
plt.legend()
plt.title(f'{title}')
plt.show()

color_cc = 'k'
range_min, range_max, norm_CC, fid_avg_CC = auto_normalize_OD_charge(OD_chrgcntrst[k_question], color_cc)
norm_charge = mpl.colors.Normalize(vmin=range_min, vmax=range_max)
fig_size = 4.5
fig, axes = plt.subplots(1, 3, figsize=(fig_size * 3, fig_size))
good = axes[0].imshow(OD_chrgcntrst[k_good], norm=norm_charge)
axes[0].set_title(f"Good scan @ {angles_pospol[k_good]:.2f}") 
fig.colorbar(good)
bad = axes[1].imshow(OD_chrgcntrst[k_bad], norm=norm_charge)
axes[1].set_title(f"Bad scan @ {angles_pospol[k_bad]:.2f}")
fig.colorbar(bad)
q = axes[2].imshow(OD_chrgcntrst[k_question], norm=norm_charge)
axes[2].set_title(f"Questionable scan @ {angles_pospol[k_question]:.2f}")
fig.colorbar(q)
plt.show()

In [ ]:
## Select different 50x50 regions to plot and calculate the mean and variance of these regions for 1 scan

## Select Scan
k = 2

# averaging areas
half_size = 40 
height, width = image_stack_pospol.shape[1:]
center_y = np.array([60, 300, 475, 60,  height // 2, 475, 60,  300, 475])
center_x = np.array([60, 60, 60,   300, width // 2,  200, 475, 475, 475])

# normalize arrays to store mean and variance over the averaging areas
mean_region = np.zeros(np.shape(center_y)[0])
variance_region = np.zeros(np.shape(center_y)[0])
mean_ua_region = np.zeros(np.shape(center_y)[0])
variance_ua_region = np.zeros(np.shape(center_y)[0])

color_cc = 'green'
range_min, range_max, norm_CC, fid_avg_CC = auto_normalize_charge(OD_chrgcntrst[k], color_cc)

for i in range(np.shape(center_y)[0]):

    # Slice
    cropped_XMCD = OD_XMCD[k][center_y[i] - half_size:center_y[i] + half_size,
                          center_x[i] - half_size:center_x[i] + half_size]
    cropped_CC = OD_chrgcntrst[k][center_y[i] - half_size:center_y[i] + half_size,
                          center_x[i] - half_size:center_x[i] + half_size]

    # ua stands for unaligned
    cropped_XMCD_ua = XMCD_unaligned[k][center_y[i] - half_size:center_y[i] + half_size,
                          center_x[i] - half_size:center_x[i] + half_size]
    cropped_CC_ua = CC_unaligned[k][center_y[i] - half_size:center_y[i] + half_size,
                          center_x[i] - half_size:center_x[i] + half_size]
    
    mean_region[i] = np.mean(cropped_CC)
    variance_region[i] = np.var(cropped_CC)
    mean_ua_region[i] = np.mean(cropped_CC_ua)
    variance_ua_region[i] = np.var(cropped_CC_ua)

    # Plot the images to see what each section looks like 
    norm_XMCD = mpl.colors.Normalize(vmin=-0.1, vmax=0.1)
    norm_charge = mpl.colors.Normalize(vmin=1, vmax=3)
    fig_size = 4.5
    fig, axes = plt.subplots(1, 2, figsize=(fig_size * 2, fig_size))
    XMCD_OD = axes[0].imshow(cropped_XMCD,cmap = 'viridis', norm=norm_XMCD)
    axes[0].set_title(f"XMCD @ {shifted_angles[k]:.2f} \n averaging area is {center_y[i]},{center_x[i]}") 
    fig.colorbar(XMCD_OD)
    Charge_OD = axes[1].imshow(cropped_CC,cmap = 'viridis', norm=norm_charge)
    axes[1].set_title(f"Charge constrast @ {shifted_angles[k]:.2f} \n averaging area is {center_y[i]},{center_x[i]}")
    fig.colorbar(Charge_OD)
    plt.show()

    
## Plot
fig_size = 5
fig, axes = plt.subplots(1, 2, figsize=(fig_size * 2, fig_size))
x = range(center_y.shape[0])
axes[0].plot(x,mean_region, label = 'aligned')
axes[0].plot(x,mean_ua_region, label = 'unaligned')
axes[0].set_title(f"Charge Contrast mean for {Gamma} \n @ {shifted_angles[k]:.2f}") 
axes[0].set_xlabel('region (arb)')
axes[0].set_ylabel('mean')
axes[0].legend()

axes[1].plot(x ,variance_region, label = 'aligned')
axes[1].plot(x ,variance_ua_region, label = 'unaligned')
axes[1].set_title(f"Charge constrast variance for {Gamma} \n @ {shifted_angles[k]:.2f}")
axes[1].set_xlabel('region (arb)')
axes[1].set_ylabel('variance')
axes[1].legend()
plt.show()

In [ ]:
## Compare Mean and Variance over all scans at a select region

## Select region in center_x an center_y from above 
i = 0

# normalize arrays to store mean and variance over the averaging areas
mean = np.zeros(np.shape(image_stack_pospol)[0])
variance = np.zeros(np.shape(image_stack_pospol)[0])
mean_ua = np.zeros(np.shape(image_stack_pospol)[0])
variance_ua = np.zeros(np.shape(image_stack_pospol)[0])
mean_sb = np.zeros(np.shape(image_stack_pospol)[0])
variance_sb = np.zeros(np.shape(image_stack_pospol)[0])


# color_cc = 'green'
# range_min, range_max, norm_CC, fid_avg_CC = auto_normalize_charge(OD_chrgcntrst[k], color_cc)

for k in range(image_stack_pospol.shape[0]):

    # Slice
    cropped_CC = OD_chrgcntrst[k][center_y[i] - half_size:center_y[i] + half_size,
                          center_x[i] - half_size:center_x[i] + half_size]

    # ua stands for unaligned
    cropped_CC_ua = CC_unaligned[k][center_y[i] - half_size:center_y[i] + half_size,
                          center_x[i] - half_size:center_x[i] + half_size]

    # # ua stands for unaligned
    # cropped_CC_sb = OD_chrgcntrst_subB[k][center_y[i] - half_size:center_y[i] + half_size,
    #                       center_x[i] - half_size:center_x[i] + half_size]

    
    mean[k] = np.mean(cropped_CC)
    variance[k] = np.var(cropped_CC)
    mean_ua[k] = np.mean(cropped_CC_ua)
    variance_ua[k] = np.var(cropped_CC_ua) 
    # mean_sb[k] = np.mean(cropped_CC_sb)
    # variance_sb[k] = np.var(cropped_CC_sb)


avg_mean = np.mean(mean)
avg_var = np.mean(variance)
## Plot
fig_size = 5.5
x = range(image_stack_pospol.shape[0])
fig, axes = plt.subplots(1, 2, figsize=(fig_size * 2.5, fig_size))
axes[0].plot(x,mean, color = 'tab:blue',  label = 'aligned')
axes[0].plot(x,mean_ua, color = 'tab:orange', label = 'unaligned')
axes[0].axhline(avg_mean, color = 'k', label = f'avg mean ={avg_mean}')
# axes[0].plot(x,mean_sb, color = 'tab:green', label = 'sub background')
axes[0].set_title(f"Charge Contrast mean for {Gamma}\n averaging area {center_y[i]},{center_x[i]}") 
axes[0].set_xlabel('index (arb)')
axes[0].set_ylabel('mean')
axes[0].legend(loc='upper left')

axes[1].plot(x,variance, color = 'tab:blue',  label = 'aligned')
axes[1].plot(x,variance_ua, color = 'tab:orange', label = 'unaligned')
axes[1].axhline(avg_var, color = 'k', label = f'avg var ={avg_var}')
# axes[1].plot(x,variance_sb, color = 'tab:green', label = 'sub background')
axes[1].set_title(f"Charge constrast variance for {Gamma}\n averaging area {center_y[i]},{center_x[i]}")
axes[1].set_xlabel('index (arb)')
axes[1].set_ylabel('variance')
axes[1].legend()
# axes[1].set_ylim(0,0.005)

plt.show()


# Quantify error with PSD

In [ ]:
## compute PSD functions

def compute_fft(image, plot): 
    #image = image.astype(np.float32)

    # Normalizing Image so we can compare all CC at each angle 
    image = (image - image.min())/(image.max() - image.min())

    # comput the multidimensional discrete Fourier transform 
    F = fftn(image) 
    # shift the zero-frequency component to the center of the spectrum 
    F_shifted = fftshift(F)
    
    psd = np.abs(F_shifted)**2

    if plot == True:
        #plot for reference
        norm = mpl.colors.Normalize(vmin=0, vmax=50)
        norm_psd = mpl.colors.Normalize(vmin=0, vmax=10000)
        fig_size = 3.5
        fig, axes = plt.subplots(1, 2, figsize=(fig_size * 3, fig_size))
        fft_plot = axes[0].imshow(F_shifted.real, cmap='hot', norm=norm)
        fig.colorbar(fft_plot)
        axes[0].set_title(f"FFT of Charge Contrast {Gamma} @ {angles_pospol[k]:.2f}")
        psd_plot = axes[1].imshow(psd, cmap='hot', norm = norm_psd)
        fig.colorbar(psd_plot)
        axes[1].set_title(f"PSD of Charge Contrast {Gamma} @ {angles_pospol[k]:.2f}")
        plt.show()
    
    return psd 

def radial_average(psd): 
    y, x = np.indices(psd.shape) 
    center = np.array([(x.max() -x.min())/2.0,(y.max() - y.min())/2.0])
    r = np.sqrt((x - center[0])**2 + (y - center[1])**2).astype(int)


    tbin = np.bincount(r.ravel(), psd.ravel())
    nr = np.bincount(r.ravel())
    radialprofile = tbin / np.maximum(nr, 1) 
    
    return radialprofile

def compare_CC(unaligned, aligned, sub_b, plot,  band = (2,10)): 
        psd_u = compute_fft(unaligned, plot)
        psd_a = compute_fft(aligned, plot)
        psd_sb = compute_fft(sub_b, plot)

        rad_u = radial_average(psd_u)
        rad_a = radial_average(psd_a)
        rad_sb = radial_average(psd_sb)

        if plot == True:
            #plot for reference
            fig_size = 3.5
            fig, axes = plt.subplots(1, 1, figsize=(fig_size * 3, fig_size))
            axes.plot(rad_u, label = 'unaligned Charge Contrast')
            axes.plot(rad_a, label = 'aligned Charge Contrast')
            axes.plot(rad_sb, label = 'background subtracted Charge Contrast')
            axes.legend()
            axes.set_title(f"Radial profile of Charge Contrast PSD {Gamma} @ {angles_pospol[k]:.2f}")
            axes.set_xlim(band)
            axes.set_ylim(0,10000)
            plt.show()
    
        fmin, fmax = band
        band_u = np.sum(rad_u[fmin:fmax])
        band_a = np.sum(rad_a[fmin:fmax])
        band_sb = np.sum(rad_sb[fmin:fmax])

        return band_u, band_a, band_sb

def generate_uniform_noise_image(shape, low, high):
    return np.random.uniform(low, high, size=shape)
    
def generate_gaussian_noise_image(shape, mean, stdev):
    'generate gaussian noise based on the variance and mean from above ' 
    return np.random.normal(loc = mean, scale = stdev, size=shape)

def PSD_Check(image, region): 
        
    ## store max values
    band = np.zeros(np.shape(image_stack_pospol)[0])
    
    ## Make a list of scans do not meet the noise criteria: 
    scans_to_check_uniform = []
    scans_to_check_gauss = []
    scans_to_check_avg = []
    
    for k in range(image_stack_pospol.shape[0]):
    
        cropped_img = image[k][center_y[region] - half_size:center_y[region] + half_size,
                              center_x[region] - half_size:center_x[region] + half_size]
    
        band[k], ____, _____ = compare_CC(cropped_img, cropped_img, cropped_img, plot = False)
    
        if band[k] > band_noise_unif: 
            scans_to_check_uniform.append(k)
        if band[k] > band_noise_gauss: 
            scans_to_check_gauss.append(k)
        if band[k] > band_noise_avg: 
            scans_to_check_avg.append(k)

    return band, scans_to_check_uniform, scans_to_check_gauss, scans_to_check_avg

    

In [ ]:
## Plot PSD for a single scan 
## Pick scan
k = 0
angle = shifted_angles[k]
i = 0
plot = True 

cropped_CC = OD_chrgcntrst[k][center_y[i] - half_size:center_y[i] + half_size,
                      center_x[i] - half_size:center_x[i] + half_size]

cropped_CC_ua = CC_unaligned[k][center_y[i] - half_size:center_y[i] + half_size,
                      center_x[i] - half_size:center_x[i] + half_size]

# cropped_CC_sb = OD_chrgcntrst_subB[k][center_y[i] - half_size:center_y[i] + half_size,
#                       center_x[i] - half_size:center_x[i] + half_size]

band_u, band_a, band_sb = compare_CC(cropped_CC_ua, cropped_CC, cropped_CC, plot)

    

In [ ]:
## Compare PSD over all scans at selected region 

## Select region 
i = 0 #(60,60)

#i = 4 # (center)
# plot = False

## Generate a random grid. Values based on aligned image mean and variance plot
noise_gauss = generate_gaussian_noise_image((half_size*2,half_size*2), avg_mean, np.sqrt(avg_var))
noise_uniform = generate_uniform_noise_image((half_size*2,half_size*2), avg_mean-np.sqrt(avg_var), avg_mean+np.sqrt(avg_var))
band_noise_unif, band_noise_gauss, __ = compare_CC(noise_uniform, noise_gauss, noise_gauss, plot = False)
band_noise_avg = np.mean(np.array([band_noise_unif, band_noise_gauss]))

## store max values
band_u = np.zeros(np.shape(image_stack_pospol)[0])
band_a = np.zeros(np.shape(image_stack_pospol)[0])
band_sb = np.zeros(np.shape(image_stack_pospol)[0])

## Make a list of scans do not meet the noise criteria: 
scans_to_check_uniform = []
scans_to_check_gauss = []
scans_to_check_avg = []

for k in range(image_stack_pospol.shape[0]):

    cropped_CC = OD_chrgcntrst[k][center_y[i] - half_size:center_y[i] + half_size,
                          center_x[i] - half_size:center_x[i] + half_size]
    
    cropped_CC_ua = CC_unaligned[k][center_y[i] - half_size:center_y[i] + half_size,
                          center_x[i] - half_size:center_x[i] + half_size]
    
    cropped_CC_sb = CC_unaligned[k][center_y[i] - half_size:center_y[i] + half_size,
                          center_x[i] - half_size:center_x[i] + half_size]

    band_u[k], band_a[k], band_sb[k] = compare_CC(cropped_CC_ua, cropped_CC, cropped_CC_sb, plot = False)

    if band_a[k] > band_noise_unif: 
        scans_to_check_uniform.append(k)
    if band_a[k] > band_noise_gauss: 
        scans_to_check_gauss.append(k)
    if band_a[k] > band_noise_avg: 
        scans_to_check_avg.append(k)


## Plot Noise
#norm = mpl.colors.Normalize(vmin=0.9, vmax=1.175)
# norm = mpl.colors.Normalize(vmin=1.07, vmax=1.25)
fig_size = 4.5
fig, axes = plt.subplots(1, 1, figsize=(fig_size * 2, fig_size))
noise = axes.imshow(noise_gauss, cmap = 'viridis')
axes.set_title(f"Noise") 
fig.colorbar(noise)
plt.show()

## Plot PSD 
fig_size = 4.5
fig, axes = plt.subplots(1, 1, figsize=(fig_size * 2, fig_size))
x = range(image_stack_pospol.shape[0])
axes.plot(x,band_a, color = 'tab:blue', label = 'aligned')
# axes.axhline(np.mean(band_a), color = 'k', label = f'mean peak value = {np.mean(band_a):.2f}')
axes.axhline(band_noise_unif, color = 'red', label = f'limit of uniform noise = {band_noise_unif:.2f}')
axes.axhline(band_noise_gauss, color = 'purple', label = f'limit of Gaussian noise = {band_noise_gauss:.2f}')
axes.axhline(band_noise_avg, color = 'tab:brown', label = f'Avg noise = {band_noise_avg:.2f}')
# axes.plot(x,band_u, color = 'tab:orange', label = 'unaligned')
#axes.plot(x,band_sb,color = 'tab:green', label = 'background subtracted')
axes.set_title(f"PSD Charge Contrast mean for {Gamma} \n averaging area {center_y[i]},{center_x[i]}") 
axes.set_xlabel('index (arb)')
axes.set_ylabel('PSD')
# axes.set_ylim(0,0.0001)
axes.legend(loc='upper right')
plt.show()

print(f'scans to check, uniform: {scans_to_check_uniform}')
print(f'scans to check, avg: {scans_to_check_avg}')
print(f'scans to check, gauss: {scans_to_check_gauss}')

In [ ]:
## look at bad scans in scans_to_check
for k in scans_to_check_uniform:
    angle = shifted_angles[k]
    print(f'k = {k}')
    color_cc = 'green'
    range_min, range_max, norm_CC, fid_avg_CC = auto_normalize_OD_charge(OD_chrgcntrst[k], color_cc)
    norm_XMCD = mpl.colors.Normalize(vmin=-0.1, vmax=0.1)
    # norm_charge = mpl.colors.Normalize(vmin=3.0, vmax=3.3)
    # norm_charge = mpl.colors.Normalize(vmin=2.3, vmax=2.6)
    # norm_charge = mpl.colors.Normalize(vmin=1.8, vmax=2.1)
    norm_charge = mpl.colors.Normalize(vmin=range_min, vmax=range_max)
    # norm_charge = mpl.colors.Normalize(vmin=1.4, vmax=1.7)
    fig_size = 5
    fig, axes = plt.subplots(1, 2, figsize=(fig_size * 2, fig_size))
    XMCD_OD = axes[0].imshow(OD_XMCD[k], norm=norm_XMCD)
    axes[0].set_title(f"XMCD @ {angle:.2f}") 
    fig.colorbar(XMCD_OD)
    Charge_OD = axes[1].imshow(OD_chrgcntrst[k], norm=norm_charge)
    axes[1].set_title(f"Charge constrast @ {angle:.2f}")
    fig.colorbar(Charge_OD)
    plt.show()
    

In [ ]:
## look at bad scans in scans_to_check
plot_EB = [20,67 ]
for k in plot_EB:
    angle = shifted_angles[k]
    print(f'k = {k}')
    color_cc = 'green'
    range_min, range_max, norm_CC, fid_avg_CC = auto_normalize_OD_charge(OD_chrgcntrst[k], color_cc)
    norm_XMCD = mpl.colors.Normalize(vmin=-0.1, vmax=0.1)
    # norm_charge = mpl.colors.Normalize(vmin=3.3, vmax=3.6)
    # norm_charge = mpl.colors.Normalize(vmin=2.45, vmax=2.75)
    # norm_charge = mpl.colors.Normalize(vmin=1.8, vmax=2.1)
    # norm_charge = mpl.colors.Normalize(vmin=1.0, vmax=1.3)
    norm_charge = mpl.colors.Normalize(vmin=range_min, vmax=range_max)
    fig_size = 4.5
    fig, axes = plt.subplots(1, 2, figsize=(fig_size * 2, fig_size))
    XMCD_OD = axes[0].imshow(OD_XMCD[k], norm=norm_XMCD)
    axes[0].set_title(f"XMCD @ {angle:.2f}") 
    fig.colorbar(XMCD_OD)
    Charge_OD = axes[1].imshow(OD_chrgcntrst[k], norm=norm_charge)
    axes[1].set_title(f"Charge constrast @ {angle:.2f}")
    fig.colorbar(Charge_OD)
    plt.show()

# Rerun scan, applying issues seen in scans to check

In [ ]:
stop

In [ ]:
print(scans_to_check_uniform)

In [ ]:
scans to check, avg: [7, 41, 44, 59, 79, 95, 96, 97, 98, 99, 100, 104, 105, 107, 108, 109, 110, 111, 115]

In [ ]:
# figure out which indices are NOT in scans_to_check_avg
valid_indices = [k for k in range(image_stack_pospol.shape[0]) 
                 if k not in scans_to_check_uniform]

# initialize arrays with correct size
CC_skipped = np.zeros((len(valid_indices),) + OD_chrgcntrst.shape[1:], 
                      dtype=OD_chrgcntrst.dtype)
XMCD_skipped = np.zeros((len(valid_indices),) + OD_XMCD.shape[1:], 
                        dtype=OD_XMCD.dtype)
angles_skipped = np.zeros((len(valid_indices),) + angles_pospol.shape[1:], 
                        dtype=OD_XMCD.dtype)
print(angles_skipped.shape)
print(angles_pospol.shape)

# fill arrays
ct = 0
for k in range(image_stack_pospol.shape[0]):
    if k in scans_to_check_avg:
        print(f'k = {k}, angle = {shifted_angles[k]:.2f}')
    else:
        CC_skipped[ct] = OD_chrgcntrst[k]
        XMCD_skipped[ct] = OD_XMCD[k]
        angles_skipped[ct] = angles_pospol[k]
        ct += 1

In [ ]:
# tifffile.imwrite(os.path.join(data_dir, f"OD_chrgcntrst_stack{date}_skipped.tiff"), CC_skipped)
# tifffile.imwrite(os.path.join(data_dir, f"OD_XMCD_stack{date}_skipped.tiff"), XMCD_skipped)
np.savetxt(os.path.join(data_dir,f"angle{date}_skipped.txt"), angles_skipped)

# Skip scans that failed average

In [ ]:
## Run full alpha scan, no background subtraction

## Initialize arrays 
translated_negpol_final = np.zeros_like(image_stack_negpol)
translated_negpol_phase_final = np.zeros_like(phase_stack_negpol)

norm_pos_OD_final = np.zeros_like(image_stack_pospol)
norm_neg_OD_final = np.zeros_like(image_stack_negpol)

aligned_error_norm_final = np.empty(image_stack_pospol.shape[0])

## Initialize other parameters
# colors needed for function :/
color_neg = 'blue'
color_pos = 'orange'
factor = 0.7 #30%

for k in range(image_stack_pospol.shape[0]):

    if k in scans_to_check_avg: 
        print('bounds = 20')
        bound = 50

    else: 
        bound = 50
    
    angle = angles_pospol[k]

    norm_neg, fid_avg_neg = auto_normalize_by_fiducial(image_stack_negpol[k],color_neg)
    norm_pos, fid_avg_pos = auto_normalize_by_fiducial(image_stack_pospol[k],color_pos)


    ## apply a mask to 30% of hole intenstity 
    masked_pos, masked_neg = mask_holes_to_one(norm_pos, norm_neg, factor)

    ## Perform crosscorrelation on normalized/masked values
    translation, corr = align_images_CC(masked_pos, masked_neg, bound)
    
    # Translating the Image
    f_target = fftn(norm_neg)
    f_target_shifted = fourier_shift(f_target, translation)
    translated_negpol_OD = ifftn(f_target_shifted).real

    # Translating the phase image by the same amount
    f_target = fftn(phase_stack_negpol[k])
    f_target_shifted = fourier_shift(f_target, translation)
    translated_negpol_phase_final[k] = ifftn(f_target_shifted).real

    
    ## XMCD
    unaligned_diffMap = -1*np.log(norm_neg) - (-1*np.log(norm_pos))
    aligned_diffMap = -1*np.log(translated_negpol_OD) - (-1*np.log(norm_pos))

    plot_XMCD_crosscorr(unaligned_diffMap, aligned_diffMap, corr)
    

    norm_pos_OD_final[k] = -1*np.log(norm_pos)
    norm_neg_OD_final[k] = -1*np.log(norm_neg)
    translated_negpol_final[k] =-1*np.log(translated_negpol_OD)

    # Evaluating the transformation's alignment by comparing the difference image before and after alignment
    align_error_mask = calc_NRMSE(translated_negpol_final[k], norm_pos_OD[k])

    aligned_error_norm_final[k] = align_error_mask
    
    print(f"Error Before = {orig_error_mask*100:.2f}%\nError After = {align_error_mask*100:.2f}%")


In [ ]:
CC_final = norm_pos_OD_final + translated_negpol_final
XMCD_final = translated_negpol_final - norm_pos_OD_final

In [ ]:
band_final, scans_to_check_final, ______, _______ = PSD_Check(CC_final, 4) 
print(scans_to_check_final)
print(len(scans_to_check_final))
print(len(scans_to_check_avg))

In [ ]:
## look at bad scans in scans_to_check
for k in scans_to_check_final:
    angle = angles_pospol[k]
    print(f'k = {k}')
    color_cc = 'green'
    range_min, range_max, norm_CC, fid_avg_CC = auto_normalize_charge(OD_chrgcntrst[k], color_cc)
    norm_XMCD = mpl.colors.Normalize(vmin=-0.1, vmax=0.1)
    norm_charge = mpl.colors.Normalize(vmin=0.5, vmax=1.5)
    fig_size = 4.5
    fig, axes = plt.subplots(1, 2, figsize=(fig_size * 2, fig_size))
    XMCD_OD = axes[0].imshow(OD_XMCD[k], norm=norm_XMCD)
    axes[0].set_title(f"XMCD @ {angle:.2f}") 
    fig.colorbar(XMCD_OD)
    Charge_OD = axes[1].imshow(OD_chrgcntrst[k], norm=norm_charge)
    axes[1].set_title(f"Charge constrast @ {angle:.2f}")
    fig.colorbar(Charge_OD)
    plt.show()

# save tiff of XMCD and charge contrast

In [ ]:
stop

In [ ]:
date

In [ ]:
Gamma

In [ ]:
## Optical Density
tifffile.imwrite(os.path.join(data_dir, f"OD_negpol_aligned_stack{date}.tiff"), translated_negpol)### CHANGE BACK TO FINAL

tifffile.imwrite(os.path.join(data_dir, f"OD_chrgcntrst_stack{date}_skipped.tiff"), CC_final)### CHANGE BACK TO FINAL
tifffile.imwrite(os.path.join(data_dir, f"OD_XMCD_stack{date}_skipped.tiff"), XMCD_final) ### CHANGE BACK TO FINAL

# tifffile.imwrite(os.path.join(data_dir, f"OD_chrgcntrst_log_stack{date}.tiff"), OD_chrgcntrst_log)
# tifffile.imwrite(os.path.join(data_dir, f"OD_XMCD_log_stack{date}.tiff"), OD_XMCD_log)

# tifffile.imwrite(os.path.join(data_dir, f"OD_chrgcntrst_sb_stack{date}_v2.tiff"), OD_chrgcntrst_subB)
# tifffile.imwrite(os.path.join(data_dir, f"OD_XMCD_sb_stack{date}_v2.tiff"), OD_XMCD_subB)

# ## Phase
# tifffile.imwrite(os.path.join(data_dir, f"phase_negpol_aligned_stack{date}_v2.tiff"), translated_negpol_phase)

# phase_chrgcntrst = phase_stack_pospol + translated_negpol_phase_final
# phase_XMCD = translated_negpol_phase_final - phase_stack_pospol

# tifffile.imwrite(os.path.join(data_dir, f"phase_chrgcntrst_stack{date}_v2.tiff"), phase_chrgcntrst)
# tifffile.imwrite(os.path.join(data_dir, f"phase_XMCD_stack{date}_v2.tiff"), phase_XMCD)

In [ ]:
stop